# Human-Vehicle Interactions from an Annotated Clip

Shows a vision-language model the **annotated** clip — the mp4 `render_tracked_video` writes, with
every person and vehicle the tracker found marked at the corners of its box and labelled `P3`, `V1`
— and asks what human-vehicle interactions are in it.

Everything the run does lives in `human_vehicle.interactions` and `human_vehicle.vlm`. This notebook
picks a clip, runs it, and shows what came back.

## The track labels an interaction carries

An interaction is one person with one vehicle, described in words and placed in time. It also
carries the **track labels** that were on those two objects:

```json
{"person_ids": ["P3", "P7"], "vehicle_ids": ["V2"], "interaction": "opens the driver-side door", ...}
```

An ideal detector-tracker would give each object exactly one label. A real one gives none, one, or
several — a person can go unlabelled for a stretch and come back under a new number. So these are
lists, in the order the labels were first seen, and an empty one is an ordinary answer: it means the
tracker missed that object, and the description is then the only handle on it.

They exist so a later step can merge the same interaction seen by two overlapping windows, when the
spans overlap and the labels agree. **Nothing here merges anything**, and nothing is de-duplicated:
at 8 s / 4 s windowing one event is usually reported twice, and the second sighting is corroboration.

The prompt tells the model the labels come from an imperfect detector and tracker, and that they are
reference only — what counts as an interaction is decided from the imagery, so an unlabelled person
handling a vehicle is reported like any other.

## Your footage leaves this machine

This uploads the annotated clip to Google. Two retention facts, both handled here:

- **Uploaded files** are kept **48 hours**, then deleted automatically. The cleanup section at the
  end removes them sooner and lists anything still stored.
- **Interactions** — the prompt, the media reference, and the model's output describing the people in
  the footage — are retained **55 days** on the paid tier *by default*. Every request sets
  `store=False` to switch that off; `human_vehicle.vlm` asserts it on each one.

## Order of use

Top to bottom. **Correctness checks** make no API calls; the **Smoke check** is one tiny call and
gates the rest. Then a whole-clip run, which is also what aims the **live timebase check** — and that
check is what unblocks the windowed runs after it. Stopping after the whole-clip run is a perfectly
good session; the check only matters if you intend to window.

## 1. Setup

In [ ]:
from __future__ import annotations

import itertools
import json
import math
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
from PIL import Image, ImageDraw, ImageFont

from human_vehicle.interactions import (
    PROMPT_VERSION,
    ClipInteractions,
    InteractionRun,
    ReportedInteraction,
    WindowRun,
    build_prompt,
    find_interactions,
    make_windows,
    probe_duration,
    run_slug,
    run_tag,
    stated_duration,
    validate_interaction,
)
from human_vehicle.merge import merge_interactions, merge_summary
from human_vehicle.vlm import GeminiBackend


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root (no pyproject.toml above the cwd).")


REPO_ROOT = find_repo_root()
# Put the clips you want to run in here yourself. Created if it does not exist, and git-ignored.
CLIPS_DIR = REPO_ROOT / "notebooks" / "outputs" / "annotated"
OUTPUT_DIR = REPO_ROOT / "notebooks" / "outputs" / "interactions"
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(REPO_ROOT / ".env")

# Constructed here but not used until a call is made: the client and the key are resolved lazily, so
# the correctness checks below run with neither.
BACKEND = GeminiBackend()

# Eight-second windows advancing four: every second is examined in a short context, and
# neighbours overlap by four, so an event shorter than that is seen whole by at least one window.
WINDOW_S = 8.0
STRIDE_S = 4.0

print(f"repo root:      {REPO_ROOT}")
print(f"clips in:       {CLIPS_DIR}")
print(f"records out:    {OUTPUT_DIR}")
print(f"backend:        {BACKEND.slug}  (prompt {PROMPT_VERSION})")
print(f"windows:        {WINDOW_S:g}s every {STRIDE_S:g}s")

## 2. The clips

Every mp4 in `CLIPS_DIR`, in name order. You fill that folder yourself; nothing here filters it or
picks a favourite.

Two things to get right when you put a clip there:

- **An annotated render, not a source clip.** The prompt tells the model to read `P3` / `V1` labels
  out of the pixels. An un-annotated clip still produces a run, and it looks like a normal one — it
  just answers a different question, with every id list empty.
- **Not a `__debug` render.** Those append each box's confidence to its label, taking it from two
  characters to eight, which is straight out of the pixel budget the overlay exists to respect.

The filename stem is the clip id the prompt is given and the folder each record is written under, so
name the files the way you want them to appear in the output.

In [ ]:
def probe(path: Path) -> dict[str, Any]:
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise RuntimeError(f"could not open {path}")
    info = {
        "path": path,
        "clip_id": path.stem,
        "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "frames": int(capture.get(cv2.CAP_PROP_FRAME_COUNT)),
        # ffprobe, not the decoder's frame count over its rate: this is the number the prompt states
        # and the number every reported time is judged against.
        "duration_s": probe_duration(path),
    }
    capture.release()
    return info


CLIPS = [probe(path) for path in sorted(CLIPS_DIR.glob("*.mp4"))]
if not CLIPS:
    raise RuntimeError(f"no mp4 files in {CLIPS_DIR} - put the annotated clips you want to run there")

for info in CLIPS:
    size = f"{info['width']}x{info['height']}"
    print(f"  {info['clip_id']:<40} {info['duration_s']:6.1f}s {size:>11} {info['frames']:>5} frames")

# The shortest clip is the cheapest thing to smoke test against. Everything below runs CLIP; point it
# at another entry to run another clip.
SMOKE = min(CLIPS, key=lambda i: i["duration_s"])
CLIP = CLIPS[0]
print(f"\nsmoke clip:   {SMOKE['clip_id']}")
print(f"running:      {CLIP['clip_id']}")

## 3. Correctness checks

No API calls. These are about the clip you are about to spend money on: that its windows tile it
with no gap, and that the contract the prompt states is the one the validator holds answers to.

The module's own behaviour is covered by `tests/`; what cannot be tested there is this clip.

In [ ]:
_duration = stated_duration(CLIP["duration_s"])
_windows = make_windows(_duration, WINDOW_S, STRIDE_S)

assert _windows[0][0] == 0.0, "the first window must start at the clip start"
assert _windows[-1][1] == _duration, "the last window must be anchored to the clip end"
for _earlier, _later in itertools.pairwise(_windows):
    assert _later[0] <= _earlier[1], f"gap between {_earlier} and {_later}"
print(
    f"  {CLIP['clip_id']}: {_duration:.1f}s -> {len(_windows)} window(s) "
    f"of {WINDOW_S:g}s/{STRIDE_S:g}s {_windows[:3]}..."
)

# The id lists: what the run is for. An empty list and several labels are both ordinary answers; a
# label nothing could have drawn is not.
_base = {
    "person_ids": ["P3", "P7"],
    "vehicle_ids": [],
    "person_description": "person in a red jacket",
    "vehicle_description": "white sedan",
    "interaction": "opens the driver-side door",
    "start_time_s": 4.0,
    "evidence_time_s": 6.0,
    "end_time_s": 8.0,
    "confidence": 0.9,
}
assert not validate_interaction(_base, _duration), "several person labels and no vehicle label is valid"
for _label, _bad in [
    ("a free-form id", {**_base, "person_ids": ["person_1"]}),
    ("the wrong category", {**_base, "person_ids": ["V2"]}),
    ("a padded number", {**_base, "person_ids": ["P01"]}),
    ("past the end of the clip", {**_base, "end_time_s": 900.0}),
]:
    _errors = validate_interaction(_bad, _duration)
    assert _errors, f"{_label} should have been rejected"
    print(f"  rejected  {_label:<24} -> {_errors[0]}")

print("correctness checks: PASS")

## 4. Smoke check

One tiny call on the shortest clip, gating everything after it. It exercises upload, polling, the
video part, schema-constrained output and `store=False` in one cheap request, so a wrong parameter
name fails here rather than midway through a batch.

In [ ]:
SMOKE_BACKEND = GeminiBackend(thinking_level="low", max_output_tokens=2048)

_smoke = find_interactions(SMOKE["path"], SMOKE_BACKEND, clip_id=SMOKE["clip_id"])
assert _smoke.error is None, f"smoke call failed: {_smoke.error}"

print(f"  clip:   {SMOKE['clip_id']} ({_smoke.duration_s:.1f}s)")
print(f"  json:   {len(_smoke.interactions)} interaction(s), {len(_smoke.malformed)} malformed")
print(
    f"  tokens: {_smoke.usage.get('total_input_tokens', 0):.0f} in, "
    f"{_smoke.usage.get('billable_output_tokens', 0):.0f} billable out"
)
print(f"  cost:   ${_smoke.usage.get('estimated_cost_usd', 0):.5f}")
for _interaction in _smoke.interactions:
    print(f"    {_interaction.person_ids} / {_interaction.vehicle_ids}  {_interaction.interaction}")
print("smoke check: PASS")

## 5. Running a clip

`run_clip` is `find_interactions` plus two notebook concerns: writing the record where the figures
can find it, and refusing a **windowed** run until the live timebase check below has passed. A
whole-clip run, like the one here, needs no such gate — there are no segment times to distrust.

Records are named `{configuration}__{timestamp}`, so neither a different configuration nor a repeat
of the same one overwrites what is already there. The repeat matters: there is no temperature in
this API and `seed` is best effort, so two runs at identical settings do differ, and comparing them
is how you tell a real difference from noise.

In [ ]:
def run_clip(
    info: dict[str, Any],
    *,
    window_s: float | None = None,
    stride_s: float | None = None,
    backend: Any = BACKEND,
    save: bool = True,
) -> InteractionRun:
    """One clip through the model, with the record written to OUTPUT_DIR."""
    if window_s is not None and not TIMEBASE_VERIFIED:
        raise RuntimeError(
            "Run the live clip-global timebase check before any windowed run. Until it passes, the "
            "assumption every windowed timestamp rests on is unverified."
        )

    run = find_interactions(info["path"], backend, window_s=window_s, stride_s=stride_s, clip_id=info["clip_id"])
    note = run.error or (
        f"{len(run.interactions)} interaction(s), {len(run.malformed)} malformed"
        + (f", {run.failed_windows} call(s) FAILED" if run.failed_windows else "")
        + f", ${run.usage.get('estimated_cost_usd', 0):.4f}"
    )
    print(f"{run.clip_id}: {note} in {run.elapsed_s:.1f}s")

    if save:
        path = OUTPUT_DIR / run.clip_id / f"{run_tag(run)}.json"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(run.model_dump_json(indent=2), encoding="utf-8")
        print(f"  saved {path.relative_to(REPO_ROOT)}")

        # Only after the run is safely on disk. The calls were paid for; the merged file is derived
        # and regenerable, and `merge_interactions` can raise on a record naming a window that does
        # not exist. That must cost the derived file, never the record of what the model said.
        if run.windowed:
            merged = merge_interactions(run)
            merged_path = path.with_name(f"{run_tag(run)}__merged.json")
            merged_path.write_text(merged.model_dump_json(indent=2), encoding="utf-8")
            print(f"  saved {merged_path.relative_to(REPO_ROOT)}  ({merge_summary(merged)})")
    return run


def summarize(runs: list[InteractionRun]) -> None:
    """One line per run. The clip comes first: a batch shares one configuration, so the slug on
    its own would be the same string on every row."""
    header = (
        f"{'clip':<22} {'run':<30} {'win':>4} {'fail':>5} {'det':>4} {'mal':>4} "
        f"{'in':>8} {'out*':>7} {'sec':>6} {'usd':>8}"
    )
    print(header)
    print("-" * len(header))
    for run in runs:
        print(
            f"{run.clip_id[:22]:<22} {run_slug(run)[:30]:<30} "
            f"{len(run.windows):>4} {run.failed_windows:>5} "
            f"{len(run.interactions):>4} {len(run.malformed):>4} "
            f"{run.usage.get('total_input_tokens', 0):>8.0f} "
            f"{run.usage.get('billable_output_tokens', 0):>7.0f} "
            f"{run.elapsed_s:>6.1f} {run.usage.get('estimated_cost_usd', 0):>8.4f}"
        )
    if len(runs) > 1:
        print("-" * len(header))
        print(
            f"{f'{len(runs)} clips':<22} {'total':<30} "
            f"{sum(len(r.windows) for r in runs):>4} {sum(r.failed_windows for r in runs):>5} "
            f"{sum(len(r.interactions) for r in runs):>4} {sum(len(r.malformed) for r in runs):>4} "
            f"{sum(r.usage.get('total_input_tokens', 0) for r in runs):>8.0f} "
            f"{sum(r.usage.get('billable_output_tokens', 0) for r in runs):>7.0f} "
            f"{sum(r.elapsed_s for r in runs):>6.1f} "
            f"{sum(r.usage.get('estimated_cost_usd', 0) for r in runs):>8.4f}"
        )

In [ ]:
run = run_clip(CLIP)
summarize([run])

## 6. Inspection

Frames here are sampled **locally, for figures only** — they are never sent anywhere; the model got
the video file. Each interaction's times map to the nearest sampled frame, giving the two views the
earlier notebooks used: a filmstrip per interaction, and a contact sheet of frames no interaction
covers.

The filmstrip header carries the reported `person_ids` and `vehicle_ids`, which is the thing to read
against the labels visible in the frames — a `V2` in the header and a `V2` in the picture is the
model reading the overlay; a header naming labels that are not there is not.

The uncovered sheet takes a confidence threshold, because the prompt invites low-confidence output
and one sprawling low-confidence record would otherwise mark a long stretch as covered and hide real
misses inside it — defeating the only view that measures recall.

In [ ]:
DISPLAY_FPS = 2.0
DISPLAY_LONG_SIDE = 640
_DISPLAY_CACHE: dict[str, tuple[list[Image.Image], list[float]]] = {}


def _label_font(size: int = 18) -> Any:
    try:
        return ImageFont.load_default(size=size)
    except TypeError:
        return ImageFont.load_default()


def get_display_frames(path: Path, duration_s: float) -> tuple[list[Image.Image], list[float]]:
    """Local frames for figures only. Nothing here is uploaded."""
    key = str(path)
    if key in _DISPLAY_CACHE:
        return _DISPLAY_CACHE[key]

    capture = cv2.VideoCapture(str(path))
    frames: list[Image.Image] = []
    timestamps: list[float] = []
    t = 0.0
    while t < duration_s:
        capture.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0)
        decoded, frame = capture.read()
        if decoded:
            actual = float(capture.get(cv2.CAP_PROP_POS_MSEC)) / 1000.0
            image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            width, height = image.size
            if max(width, height) > DISPLAY_LONG_SIDE:
                scale = DISPLAY_LONG_SIDE / max(width, height)
                image = image.resize((round(width * scale), round(height * scale)), Image.Resampling.LANCZOS)
            draw = ImageDraw.Draw(image)
            text = f"#{len(frames)}  {actual:.2f}s"
            box = draw.textbbox((8, 8), text, font=_label_font())
            draw.rectangle((box[0] - 5, box[1] - 5, box[2] + 5, box[3] + 5), fill="black")
            draw.text((8, 8), text, fill="white", font=_label_font())
            frames.append(image)
            timestamps.append(actual)
        t += 1.0 / DISPLAY_FPS
    capture.release()

    _DISPLAY_CACHE[key] = (frames, timestamps)
    return frames, timestamps


def nearest_frame(t: float, timestamps: list[float]) -> int:
    return min(range(len(timestamps)), key=lambda i: abs(timestamps[i] - t))


def _grid(images, titles, cols, title, save_to, display=True) -> None:
    if not images:
        return
    cols = max(1, cols)
    rows = math.ceil(len(images) / cols)
    figure, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.4 * rows))
    axes = np.atleast_1d(np.asarray(axes)).ravel()
    for axis in axes:
        axis.axis("off")
    # strict=False: the grid is padded to a full rectangle, so there are usually more axes
    # than images.
    for axis, image, subtitle in zip(axes, images, titles, strict=False):
        axis.imshow(image)
        axis.set_title(subtitle, fontsize=8)
        axis.axis("off")
    figure.suptitle(title, fontsize=10)
    figure.tight_layout()
    if save_to is not None:
        save_to.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(save_to, dpi=110, bbox_inches="tight")
    if display:
        plt.show()
    else:
        plt.close(figure)


def _ids(labels: list[str]) -> str:
    return ",".join(labels) if labels else "no id"

In [ ]:
def window_of(run: InteractionRun, index: int) -> WindowRun | None:
    return next((window for window in run.windows if window.window_index == index), None)


def interaction_frames(run: InteractionRun, item: ReportedInteraction, timestamps: list[float]) -> set[int]:
    """Frames an interaction claims, intersected with the window that reported it.

    A windowed record may legitimately report a span running past its segment, but coverage answers
    "what was examined and claimed", so a window is credited only for what it saw. Without this, an
    over-broad span from one window could mark frames a failed window never looked at as covered.
    """
    frames = set(range(nearest_frame(item.start_time_s, timestamps), nearest_frame(item.end_time_s, timestamps) + 1))
    window = window_of(run, item.window_index)
    if window is not None:
        frames &= {i for i, t in enumerate(timestamps) if window.start_s <= t <= window.end_s}
    return frames


def unexamined_frames(run: InteractionRun, timestamps: list[float]) -> set[int]:
    """Frames in a failed call's window and in no successful one.

    Narrower than the failed window's whole span: windows overlap, so a neighbour usually reclaims
    most of it, and marking the whole span would claim examined frames were not.
    """

    def span(window: WindowRun) -> set[int]:
        return {i for i, t in enumerate(timestamps) if window.start_s <= t <= window.end_s}

    failed: set[int] = set()
    for window in run.windows:
        if window.error:
            failed |= span(window)
    for window in run.windows:
        if not window.error:
            failed -= span(window)
    return failed


def show_interactions(run: InteractionRun, *, max_frames: int = 6, save: bool = True, display: bool = True) -> None:
    """One filmstrip per interaction, in clip order, each headed by the labels it reported."""
    frames, timestamps = get_display_frames(Path(run.source), run.duration_s)
    items = sorted(run.interactions, key=lambda i: (i.start_time_s, i.confidence))
    if not items:
        print(f"{run.clip_id}: no interactions")
        return
    save_dir = OUTPUT_DIR / run.clip_id / run_tag(run) if save else None

    for number, item in enumerate(items):
        evidence = nearest_frame(item.evidence_time_s, timestamps)
        span = sorted(interaction_frames(run, item, timestamps)) or [evidence]
        if len(span) > max_frames:
            step = (len(span) - 1) / max(1, max_frames - 1)
            span = sorted({span[round(i * step)] for i in range(max_frames)})
            if evidence not in span:
                nearest = min(span, key=lambda f: abs(f - evidence))
                span = sorted({*(f for f in span if f != nearest), evidence})
        header = (
            f"{_ids(item.person_ids)} / {_ids(item.vehicle_ids)}  |  conf {item.confidence:.2f}  |  "
            f"{item.interaction}\n{item.person_description} / {item.vehicle_description}  |  "
            f"{item.start_time_s:.2f}-{item.end_time_s:.2f} s"
        )
        _grid(
            [frames[i] for i in span],
            [f"#{i}{' <- evidence' if i == evidence else ''}" for i in span],
            max_frames,
            header,
            save_dir / f"interaction_{number:02d}.png" if save_dir else None,
            display=display,
        )


def show_uncovered(
    run: InteractionRun, *, min_confidence: float = 0.0, cols: int = 8, save: bool = True, display: bool = True
) -> None:
    """Frames no interaction claims. This is the miss surface."""
    frames, timestamps = get_display_frames(Path(run.source), run.duration_s)
    covered = {
        frame
        for item in run.interactions
        if item.confidence >= min_confidence
        for frame in interaction_frames(run, item, timestamps)
    }
    unexamined = unexamined_frames(run, timestamps)
    missing = [i for i in range(len(frames)) if i not in covered]

    # Never-examined frames are an API failure, not a model miss, so they are counted apart from the
    # recall number rather than folded into it.
    never_examined = unexamined & set(missing)
    line = (
        f"{run.clip_id}: {len(covered)}/{len(frames)} frames covered at confidence >= "
        f"{min_confidence:.2f}, {len(missing) - len(never_examined)} uncovered by the model"
    )
    if run.failed_windows:
        line += f"  [{run.failed_windows} call(s) FAILED, {len(never_examined)} frame(s) never examined]"
    print(line)
    if not missing:
        return

    save_dir = OUTPUT_DIR / run.clip_id / run_tag(run) if save else None
    _grid(
        [frames[i] for i in missing],
        [f"#{i}" + ("  NOT EXAMINED" if i in unexamined else "") for i in missing],
        min(cols, len(missing)),
        f"Uncovered frames (confidence >= {min_confidence:.2f})",
        save_dir / f"uncovered_conf{min_confidence:.2f}.png" if save_dir else None,
        display=display,
    )

In [ ]:
show_interactions(run)

In [ ]:
show_uncovered(run, min_confidence=0.0)
show_uncovered(run, min_confidence=0.5)

### The labels that came back

The id lists, laid out beside the times. This is what a later merge step will key on: two records
from adjacent windows whose spans overlap and whose labels agree are one interaction seen twice.

A row with `no id` on either side is the case the design exists to tolerate — the tracker missed that
object, and only the description identifies it.

In [ ]:
def show_labels(run: InteractionRun) -> None:
    header = f"{'win':>3} {'start':>7} {'end':>7} {'conf':>5}  {'person':<14} {'vehicle':<14} interaction"
    print(header)
    print("-" * len(header))
    for item in sorted(run.interactions, key=lambda i: (i.start_time_s, i.window_index)):
        print(
            f"{item.window_index:>3} {item.start_time_s:>7.2f} {item.end_time_s:>7.2f} "
            f"{item.confidence:>5.2f}  {_ids(item.person_ids):<14} {_ids(item.vehicle_ids):<14} "
            f"{item.interaction[:48]}"
        )
    unlabelled = sum(1 for i in run.interactions if not i.person_ids or not i.vehicle_ids)
    print(f"\n{len(run.interactions)} interaction(s), {unlabelled} with an object the tracker never labelled")


show_labels(run)

## 7. Live check: segment times are clip-global

The windowed path assumes the model reports times on the **clip's** clock even when it is sent only
a segment. `find_interactions` says so in its docstring and validates every record against it, but
neither can prove a model obeys — and if a model or SDK update broke it, every windowed timestamp
would shift while looking entirely plausible.

One segment call, built exactly like a windowed one — same prompt, same schema, same offsets — and
put through **the same validator the windowed path uses**. Reusing the contract rather than
restating it is the point: a looser check, or an altered request, would pass exactly the answers
that violate the assumption.

It runs here, after the whole-clip pass, because that pass says where to aim: a segment holding
nothing proves nothing, so the window is centred on the interaction the whole-clip run was surest
about. If that run found nothing at all, the check falls back to the end of the clip and will
probably have nothing to work with — which leaves windowed runs blocked rather than failing.

What does fail loudly is a time that breaks the contract, because that means every windowed
timestamp is wrong.

In [ ]:
TIMEBASE_VERIFIED = False  # only the check below sets this

# Aimed at a stretch the whole-clip run just found something in. Checking an empty segment proves
# nothing, and picking the window arithmetically is how you end up doing that by accident.
if run.interactions:
    _tb_anchor = max(run.interactions, key=lambda i: i.confidence).evidence_time_s
    _tb_start = max(0.0, min(_tb_anchor - WINDOW_S / 2, _duration - WINDOW_S))
else:
    _tb_start = max(0.0, _duration - WINDOW_S)
_tb_window = (round(_tb_start, 1), round(min(_tb_start + WINDOW_S, _duration), 1))

if TIMEBASE_VERIFIED:
    print("live check: not applicable - this backend corrects segment times itself, so windowed runs are unblocked")

_tb_response = (
    None
    if TIMEBASE_VERIFIED
    else BACKEND.generate(
        CLIP["path"],
        build_prompt(CLIP["clip_id"], _duration, _tb_window),
        window=_tb_window,
        # The same schema the windowed path sends. A check that alters the request is not checking the
        # request: unconstrained, the model answers with a bare array and the shape below does not hold.
        schema=ClipInteractions.model_json_schema(),
    )
)
_tb_items = json.loads(_tb_response.text).get("interactions", [])
print(f"  segment {_tb_window[0]:g}-{_tb_window[1]:g}s returned {len(_tb_items)} interaction(s)")
for _item in _tb_items:
    print(f"    {_item['start_time_s']:6.2f} - {_item['end_time_s']:6.2f}s  ev={_item['evidence_time_s']:.2f}")

for _item in _tb_items:
    _errors = validate_interaction(_item, _duration, _tb_window, BACKEND.tolerance_s)
    assert not _errors, (
        f"segment times are NOT clip-global any more, or violate the windowed contract: {_errors}. "
        "Every windowed timestamp would be wrong; stop and re-verify before running a sweep."
    )

# An empty segment is an ordinary answer, not an error - there may genuinely be nothing there. It
# just leaves nothing to check, so the assumption stays unverified and windowed runs stay blocked.
TIMEBASE_VERIFIED = bool(_tb_items)
if TIMEBASE_VERIFIED:
    print("live check (clip-global timebase): PASS - windowed runs unblocked")
else:
    print(
        "live check (clip-global timebase): nothing to check - this segment holds no interaction.\n"
        "  Windowed runs stay blocked. Point _tb_window at a stretch that does hold one and re-run."
    )

## 8. Windowed runs

`WINDOW_S` / `STRIDE_S` — eight seconds advancing four — so the clip is examined a segment at a
time and neighbours overlap by four. This is the mode the work is heading for: the overlap is what
gives two independent sightings of an event on a seam, and the `person_ids` / `vehicle_ids` are what
section 10 merges those sightings on.

Slower than the whole-clip pass above, a call per window, and **the counts are not comparable with
it**: one event reported by two windows is two rows here. Read a windowed run against the whole-clip
one with the **uncovered sheet** and the filmstrips, never by counting rows.

Shorter contexts also ask far less of the model per call, which is the structural fix for a long
clip exhausting its token budget on thinking and truncating its answer mid-JSON.

The live timebase check above must have passed, or this refuses to start.

In [ ]:
windowed_run = run_clip(CLIP, window_s=WINDOW_S, stride_s=STRIDE_S)
summarize([run, windowed_run])

In [ ]:
show_labels(windowed_run)
show_uncovered(windowed_run, min_confidence=0.0)

## 9. The whole folder

Every clip in `CLIPS_DIR`, one after another, each leaving a record and its figures on disk. The
figures are written rather than displayed — a folder's worth of filmstrips inline is unreadable —
and land beside the record under `OUTPUT_DIR/<clip_id>/`.

Two arms over the same folder, one cell each, each writing its own records — the windowing is in
the slug (`__w8s4__`, or absent), so nothing overwrites anything and you can re-read both later.

| arm | overlap | events always contained | calls over 8 clips |
| --- | --- | --- | --- |
| 8 s / 4 s | 4 s | under 4 s | 28 |
| whole clip | — | all of them | 8 |

Containment is decided by the **overlap**, `window - stride`, not by the window length: an event can
always begin an instant before a window ends, so what saves it is a neighbour that started early
enough to hold it whole. Chasing containment much further is not worth it — it needs overlap on the
order of the event itself, which converges on feeding the whole clip.

**Do not compare these by detection count.** More overlap reports the same event more often, so the
count rises without anything being found. Compare the **uncovered sheets** at the same confidence,
which duplicates do not affect, and the share of records whose span touches both window edges, which
is the stitching burden the windowed arm leaves for the merge. Each call costs about $0.008, so an arm
over eight clips is roughly a quarter, and these are worth repeating once — repeat runs at identical
settings already vary by a detection or two, and there is no temperature here to pin.

A clip whose calls all failed keeps its row in the table with its error, so a batch cannot hide one
bad clip among good ones.

In [ ]:
def run_all_clips(
    clips: list[dict[str, Any]] = CLIPS,
    *,
    window_s: float | None = None,
    stride_s: float | None = None,
    backend: Any = BACKEND,
    display: bool = False,
) -> list[InteractionRun]:
    """Run every clip, writing a record and figures for each. Returns the runs in folder order.

    `backend` is threaded through rather than left to `run_clip`'s default: a batch is the worst
    place to discover that calls went to a model you did not mean to use.
    """
    if window_s is not None and stride_s is not None:
        calls = sum(len(make_windows(stated_duration(info["duration_s"]), window_s, stride_s)) for info in clips)
        print(f"{len(clips)} clip(s) at {window_s:g}s/{stride_s:g}s windows -> about {calls} call(s)\n")

    runs: list[InteractionRun] = []
    for info in clips:
        run = run_clip(info, window_s=window_s, stride_s=stride_s, backend=backend)
        # Figures only where there is something to draw them from: a wholly failed run has nothing,
        # and asking for its filmstrips would just raise in the middle of a batch.
        if run.error is None:
            show_interactions(run, display=display)
            show_uncovered(run, min_confidence=0.0, display=display)
        runs.append(run)
    return runs

In [ ]:
# Arm 1 - 8s windows every 4s. The overlap is 4s, so any event shorter than that is seen whole by
# at least one window; that is what gives section 10's merge two sightings to collapse, and what
# keeps direction right, since "entering" and "exiting" look alike in a truncated view.
runs_windowed = run_all_clips(window_s=WINDOW_S, stride_s=STRIDE_S)
summarize(runs_windowed)

In [ ]:
# Arm 2 - the whole clip in one call each. No windows, so no duplicates and nothing to stitch, but
# the weaker recall of the two and the long clips must fit a whole answer in one response. Expect
# the 27s clip to fail on `max_output_tokens` at the default 8192 - thinking eats the budget and the
# JSON is cut mid-string. GeminiBackend(max_output_tokens=32768) is the fix if you want it to pass.
runs_whole = run_all_clips()
summarize(runs_whole)

## 10. What merged

Each windowed run above also wrote `{run_tag}__merged.json` beside its own record: the same events,
with each window's sighting of one event collapsed into a single row. Two records are one event when
their spans overlap **and** their person ids intersect **and** their vehicle ids intersect.

`records -> events` is the number that says whether the merge did anything. A clip where they are
equal had no duplicates to collapse -- or, more often, had duplicates the ids could not confirm.

**A record with no person id, or no vehicle id, never merges.** That is deliberate and it costs
recall: an event whose person the tracker lost stays split across windows. The alternative is worse.
One clip here has an unlabelled person entering a car's driver seat while a labelled one exits its
passenger side, overlapping in time at the same vehicle -- merging on the vehicle alone would fuse
them into a person who did both at once, and nothing downstream could tell.

The merged file is **derived**: regenerate it from the run beside it at any time. A nonzero
`failed_windows` or `malformed_count` means the clip was never fully examined, however complete the
merged list looks.

In [ ]:
def show_merges(runs: list[InteractionRun]) -> None:
    """What each run's merge did, and what it left alone."""
    for run in runs:
        if not run.windowed:
            continue
        merged = merge_interactions(run)
        print(merge_summary(merged))
        for event in merged.interactions:
            if event.sightings > 1:
                print(
                    f"    {event.start_time_s:5.1f}-{event.end_time_s:5.1f}  x{event.sightings} "
                    f"windows {event.window_indexes}  p={event.person_ids} v={event.vehicle_ids}"
                )


# Four seconds of overlap means most events are contained whole, so the merge has little left to
# stitch. `records -> events` is what says whether it collapsed anything at all.
show_merges(runs_windowed)

## 11. Cleanup

Uploaded clips expire on their own after 48 hours. This removes them now.

Run it when you are done iterating — not between calls, since reusing an uploaded file is why this
uses the File API at all.

`delete_uploads()` removes what **these backends** uploaded. The listing is the important half:
restarting the kernel loses that record, so uploads from earlier sessions stay until they expire.
Anything listed that you do not want there goes with
`BACKEND.client.files.delete(name="files/...")`.

In [ ]:
for _file in BACKEND.client.files.list():
    print(f"  stored: {_file.name}  ({getattr(_file, 'size_bytes', '?')} bytes)")

In [ ]:
for _backend in (BACKEND, SMOKE_BACKEND):
    for _name in _backend.delete_uploads():
        print(f"  deleted {_name}")

_remaining = list(BACKEND.client.files.list())
print(f"remaining: {len(_remaining)} file(s)")
for _file in _remaining:
    # Anything still here came from an earlier kernel; remove it with files.delete(name=...).
    print(f"  {_file.name}")